# Trabajo de investigación aplicado · Limpieza y generación de datos en Python

**Minería de Datos (IEI-097)** · IP Santo Tomás Chillán · Unidad I · Individual · Formativo

El **01/09** usted limpió los CSV crudos de **Supermercados Diguillín** en Excel: corrigió texto,
rellenó faltantes, quitó duplicados, filtró devoluciones, cruzó tablas con `BUSCARV`, derivó el
día de la semana y graficó. Hoy hace **exactamente el mismo trabajo, sobre los mismos archivos,
en Python** — la herramienta con la que siguen los laboratorios del curso.

**La diferencia con un laboratorio guiado:** este notebook trae la **pregunta**, las **pistas** y
el **número que debe salir**. El código lo investiga usted: documentación de `pandas`, sus
compañeros, la IA que quiera usar (declarándola). Y deja constancia de cada decisión en la
**bitácora** de cada sección.

| | |
|---|---|
| **Se lanza** | martes 22/09 (laboratorio) |
| **Se entrega** | martes 29/09, al inicio del laboratorio · `Archivo → Descargar → Descargar .ipynb` → Aula Virtual |
| **Cómo se revisa** | ese día, revisión entre pares: otro compañero ejecuta su notebook y lee su bitácora |

> **Cómo empezar:** `Archivo → Guardar una copia en Drive` (si no, pierde su trabajo al cerrar).
> Ejecute la **celda 0** primero, siempre. Las celdas se ejecutan con `Shift+Enter`.

## Celda 0 · Configuración (ya escrita — solo ejecútela)

Carga los dos archivos crudos directo desde internet y define `verificar()`: la función que le dice
si el número que obtuvo es el que debía salir. **No la modifique.**

In [ ]:
# ============================================================
#  CELDA 0 · ejecutar primero, siempre
# ============================================================
import os
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# Los CSV crudos están publicados: se leen desde la URL, no hay que subir nada.
RUTA_DATOS = "https://raw.githubusercontent.com/de-mag-ia/diguillin-datos/main/crudo/"
clientes = pd.read_csv(RUTA_DATOS + "clientes.csv")
boletas  = pd.read_csv(RUTA_DATOS + "boletas.csv")

# Los números que deben salir en cada paso (los mismos que vio en Excel el 01/09).
ESPERADO = {
    'filas_clientes': 1245,
    'filas_boletas': 57527,
    'comunas_distintas': 21,
    'medios_pago_distintos': 4,
    'comunas_vacias': 87,
    'canal_alta_vacios': 49,
    'faltantes_despues': 0,
    'comuna_sin_dato': 87,
    'rut_repetidos_exactos': 30,
    'rut_repetidos_normalizados': 45,
    'clientes_finales': 1200,
    'boletas_repetidas_por_id': 60,
    'boletas_finales': 57467,
    'devoluciones': 70,
    'boletas_ventas': 57397,
    'boletas_sin_tarjeta': 22812,
    'huerfanas': 90,
    'dia_mas_boletas': 'sábado',
    'boletas_sabado': 12658,
    'edad_mediana': 40,
    'archivo_guardado': True,
    'huerfanas_keep_first': 90,
    'huerfanas_keep_last': 1730,
    'boletas_de_fichas_eliminadas': 1640,
    'fechas_imposibles': 18,
}

_resultados = {}

def verificar(clave, valor):
    """Compara su resultado con el esperado e imprime ✅ o ❌."""
    esperado = ESPERADO[clave]
    if hasattr(valor, "item"):          # numpy → Python
        valor = valor.item()
    ok = (valor == esperado)
    _resultados[clave] = ok
    if ok:
        print(f"✅ {clave} = {valor}")
    else:
        print(f"❌ {clave} = {valor}   (debía salir {esperado})")
    return ok

def resumen():
    """Cuántas verificaciones están en ✅."""
    total, ok = len(ESPERADO), sum(_resultados.values())
    pendientes = [k for k in ESPERADO if not _resultados.get(k)]
    print(f"{ok} de {total} verificaciones en ✅")
    if pendientes:
        print("Pendientes o con ❌:", ", ".join(pendientes))

print("clientes:", clientes.shape, "  boletas:", boletas.shape)
verificar("filas_clientes", len(clientes))
verificar("filas_boletas", len(boletas))

> Si las dos líneas de arriba salieron con ✅, está listo. Si sale un error de conexión, vuelva a
> ejecutar la celda; si persiste, avise.

**Antes de limpiar, mirar** (igual que en Excel, donde primero ordenó y filtró). Ejecute las dos
celdas siguientes y observe: ¿qué problemas reconoce de los que ya vio el 01/09?

In [ ]:
clientes.head(10)

In [ ]:
boletas.head(10)

## Sección 1 · Corregir texto y categorías
*En Excel: `ESPACIOS`, `NOMPROPIO`, y `Ctrl+H` para los reemplazos dirigidos (guía de Excel, §4).*

`comuna` en `clientes` tiene **69 formas de escribir** unas pocas comunas reales (`CHILLÁN`,
`Chillan`, `chillán `…). `medio_pago` en `boletas` tiene **13 formas** de escribir **4** medios de
pago: *Débito, Efectivo, Crédito, Tarjeta Vecino*.

**Debe quedar:** `comuna` con **21 valores distintos** y `medio_pago` con
**4**. Modifique las columnas dentro de los mismos DataFrames
`clientes` y `boletas`.

In [ ]:
# SECCIÓN 1 · Corregir texto: comuna (clientes) y medio_pago (boletas)
#
# Pistas (qué buscar, no qué escribir):
#   · `clientes.comuna.value_counts()` para ver las variantes; lo mismo con `boletas.medio_pago`.
#   · Las funciones de texto de pandas viven en `.str`: busque `.str.strip()` y `.str.title()`.
#   · Después de strip + title quedan 24 comunas, no 21: las tres que sobran son tildes
#     (Chillan / Chillán…). Eso se arregla con un diccionario y `.replace({...})`.
#   · Para medio_pago, construya el diccionario {variante: canónico} con las 13 variantes
#     y aplíquelo con `.map(diccionario)` o `.replace(diccionario)`. Compruebe que no quedó NaN.

# 👉 Su código aquí

In [ ]:
# ✔ Verificación · Sección 1
verificar("comunas_distintas", clientes.comuna.nunique())
verificar("medios_pago_distintos", boletas.medio_pago.nunique())

> ✍️ **Bitácora · Sección 1**
>
> - **Decisión que tomé:** …
> - **Por qué:** …
> - **Fuente consultada** (documentación, un compañero, IA — cuál y para qué): …
> - **Qué no funcionó a la primera:** …

## Sección 2 · Rellenar valores faltantes
*En Excel: `Ir a Especial → Celdas en blanco`, y luego `MODA.UNO` o escribir `"Sin dato"` (§5).*

Dos columnas de `clientes` tienen vacíos, y en Excel les dio **tratamientos distintos**. Repítalos:

- `comuna` → marcar explícitamente con `"Sin dato"` (no se puede adivinar dónde vive alguien).
- `canal_alta` → rellenar con el valor más frecuente (la **moda**).

**Antes de rellenar, cuente** cuántos vacíos hay en cada una. **Después**, no debe quedar ninguno.

In [ ]:
# SECCIÓN 2 · Faltantes en clientes: comuna → "Sin dato", canal_alta → moda
#
# Pistas (qué buscar, no qué escribir):
#   · `.isna().sum()` cuenta vacíos por columna; guarde los dos conteos ANTES de rellenar
#     en `comunas_vacias` y `canal_alta_vacios`.
#   · Busque `.fillna()`. La moda de una columna es `.mode()[0]` (¿por qué el `[0]`?).
#   · Recuerde el orden de la guía: primero se corrigió el texto (Sección 1), después se imputa.

# 👉 Su código aquí
comunas_vacias = ...
canal_alta_vacios = ...

In [ ]:
# ✔ Verificación · Sección 2
verificar("comunas_vacias", comunas_vacias)
verificar("canal_alta_vacios", canal_alta_vacios)
verificar("faltantes_despues", clientes[["comuna", "canal_alta"]].isna().sum().sum())
verificar("comuna_sin_dato", (clientes.comuna == "Sin dato").sum())

> ✍️ **Bitácora · Sección 2**
>
> - **Decisión que tomé:** …
> - **Por qué:** …
> - **Fuente consultada** (documentación, un compañero, IA — cuál y para qué): …
> - **Qué no funcionó a la primera:** …

## Sección 3 · Quitar duplicados
*En Excel: `Datos → Quitar duplicados`, eligiendo qué columnas definen el duplicado (§6).*

**Clientes:** la empresa dice tener 1.200 socios y el archivo trae 1.245
filas. Hay personas cargadas dos veces con el RUT escrito distinto (`90.147.265-3` y
`90147265-3`). Cree una columna `rut_norm` **sin puntos, sin guion y en mayúsculas**, y cuente
duplicados por `rut` tal cual y por `rut_norm`. Luego elimine los duplicados por `rut_norm`
**conservando la primera aparición**.

**Boletas:** hay boletas repetidas (`B0001593` aparece dos veces). Elimine los duplicados por
`id_boleta`, conservando la primera.

**Debe quedar:** 1.200 clientes y 57.467 boletas.

In [ ]:
# SECCIÓN 3 · Duplicados: clientes por rut_norm, boletas por id_boleta
#
# Pistas (qué buscar, no qué escribir):
#   · `.str.replace(".", "", regex=False)` quita un carácter; encadene dos y `.str.upper()`.
#   · `.duplicated().sum()` cuenta las filas repetidas de una columna (guarde los dos conteos en
#     `rut_repetidos_exactos` y `rut_repetidos_normalizados`; el de boletas en
#     `boletas_repetidas_por_id`).
#   · Busque `drop_duplicates(subset=..., keep=...)`. Lea qué hace `keep` — vuelve en la Pregunta G1.

# 👉 Su código aquí
rut_repetidos_exactos = ...
rut_repetidos_normalizados = ...
boletas_repetidas_por_id = ...

In [ ]:
# ✔ Verificación · Sección 3
verificar("rut_repetidos_exactos", rut_repetidos_exactos)
verificar("rut_repetidos_normalizados", rut_repetidos_normalizados)
verificar("clientes_finales", len(clientes))
verificar("boletas_repetidas_por_id", boletas_repetidas_por_id)
verificar("boletas_finales", len(boletas))

> ✍️ **Bitácora · Sección 3**
>
> - **Decisión que tomé:** …
> - **Por qué:** …
> - **Fuente consultada** (documentación, un compañero, IA — cuál y para qué): …
> - **Qué no funcionó a la primera:** …

## Sección 4 · Filtrar filas problemáticas
*En Excel: filtro automático sobre `total`, `< 0`, y copiar a otra hoja (§7).*

Hay boletas con **total negativo**: son devoluciones cargadas como si fueran ventas. Sepárelas
en un DataFrame `devoluciones` y deje en `boletas` solo las ventas (`total >= 0`).

In [ ]:
# SECCIÓN 4 · Separar devoluciones (total < 0) de las ventas
#
# Pistas (qué buscar, no qué escribir):
#   · Un filtro en pandas es una condición entre corchetes: `boletas[boletas.total < 0]`.
#   · Guarde las devoluciones aparte ANTES de sobrescribir `boletas`: si primero filtra, las pierde.

# 👉 Su código aquí
devoluciones = ...

In [ ]:
# ✔ Verificación · Sección 4
verificar("devoluciones", len(devoluciones))
verificar("boletas_ventas", len(boletas))

> ✍️ **Bitácora · Sección 4**
>
> - **Decisión que tomé:** …
> - **Por qué:** …
> - **Fuente consultada** (documentación, un compañero, IA — cuál y para qué): …
> - **Qué no funcionó a la primera:** …

## Sección 5 · Cruzar dos tablas
*En Excel: `BUSCARV(...; FALSO)` envuelto en `SIERROR(...; "NO EXISTE")` (§8).*

Traiga a cada boleta el `nombre` y la `comuna` de su cliente. Guarde el resultado en un
DataFrame `cruce`. Luego cuente dos cosas distintas:

- **boletas sin tarjeta**: `id_cliente` vacío en la boleta (compra anónima, no es un error);
- **boletas huérfanas**: tienen un `id_cliente`, pero **ese cliente no existe** en `clientes`
  (el `"NO EXISTE"` de Excel). Guárdelas en `huerfanas`.

In [ ]:
# SECCIÓN 5 · Cruzar boletas con clientes y encontrar huérfanas
#
# Pistas (qué buscar, no qué escribir):
#   · Busque `merge(..., on="id_cliente", how="left")`. ¿Por qué `left` y no `inner`?
#   · Tras un cruce izquierdo, una boleta huérfana tiene `id_cliente` con valor pero `nombre` NaN.
#   · Combine dos condiciones con `&` y paréntesis: `cruce[(cond1) & (cond2)]`.

# 👉 Su código aquí
cruce = ...
huerfanas = ...

In [ ]:
# ✔ Verificación · Sección 5
verificar("boletas_sin_tarjeta", cruce.id_cliente.isna().sum())
verificar("huerfanas", len(huerfanas))

> ✍️ **Bitácora · Sección 5**
>
> - **Decisión que tomé:** …
> - **Por qué:** …
> - **Fuente consultada** (documentación, un compañero, IA — cuál y para qué): …
> - **Qué no funcionó a la primera:** …

## Sección 6 · Derivar atributos nuevos
*En Excel: `=TEXTO(A2;"dddd")` y `=SIFECHA(...)` (§9).*

**Generar datos es derivar, no inventar.** Cree dos columnas nuevas:

- en `cruce`, `dia_semana` a partir de `fecha_hora`, **en español y en minúsculas**
  (`"lunes"`, …, `"domingo"`);
- en `clientes`, `edad` a partir de `fecha_nacimiento`, calculada como **2025 − año de nacimiento**.

Luego responda: ¿qué día de la semana tiene más boletas, y cuántas?

In [ ]:
# SECCIÓN 6 · Derivar dia_semana (cruce) y edad (clientes)
#
# Pistas (qué buscar, no qué escribir):
#   · `fecha_hora` llegó como texto: conviértala con `pd.to_datetime(...)`.
#   · Una fecha tiene `.dt.dayofweek` (0 = lunes … 6 = domingo) y `.dt.year`.
#   · Para pasar de 0–6 a nombres: una lista ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"] y `.map(...)`.
#   · `.value_counts()` cuenta; `.idxmax()` dice cuál es el mayor.
#   · En `fecha_nacimiento` hay valores raros: use `errors="coerce"` y mire `.describe()` de edad.

# 👉 Su código aquí

In [ ]:
# ✔ Verificación · Sección 6
verificar("dia_mas_boletas", cruce.dia_semana.value_counts().idxmax())
verificar("boletas_sabado", (cruce.dia_semana == "sábado").sum())
verificar("edad_mediana", int(clientes.edad.median()))

> ✍️ **Bitácora · Sección 6**
>
> - **Decisión que tomé:** …
> - **Por qué:** …
> - **Fuente consultada** (documentación, un compañero, IA — cuál y para qué): …
> - **Qué no funcionó a la primera:** …

## Sección 7 · Representar patrones: gráficos
*En Excel: `Insertar → Gráfico de columnas`, `Histograma`, `Caja y bigotes` (§10).*

Dos gráficos, en dos celdas:

1. **Columnas:** boletas por `dia_semana`, en orden lunes → domingo (no en orden de frecuencia).
2. **Uno a su elección** sobre `cruce` o `clientes`: histograma de `total`, caja y bigotes de
   `total`, columnas por `medio_pago`… El que llevaría a un informe para la gerencia.

> Esto **no es minería de datos**: describe lo que ya está en los datos. Es el hallazgo simple
> que pidió la Parte C de la exposición.

In [ ]:
# SECCIÓN 7 · Gráfico 1: boletas por día de la semana, lunes → domingo
#
# Pistas (qué buscar, no qué escribir):
#   · `.value_counts()` ordena por frecuencia; `.reindex(dias)` lo pone en el orden de la lista.
#   · Una Serie se grafica con `.plot.bar(title=..., rot=0)`; `plt.show()` lo muestra.

# 👉 Su código aquí

In [ ]:
# SECCIÓN 7 · Gráfico 2: a su elección (histograma, caja y bigotes, columnas...)
#
# Pistas (qué buscar, no qué escribir):
#   · Busque `.plot.hist(bins=...)`, `.plot.box()`, `.plot.bar()`.
#   · Un gráfico sin título ni etiqueta de eje no sirve en un informe.

# 👉 Su código aquí

> ✍️ **Bitácora · Sección 7**
>
> - **Decisión que tomé:** …
> - **Por qué:** …
> - **Fuente consultada** (documentación, un compañero, IA — cuál y para qué): …
> - **Qué no funcionó a la primera:** …

## Sección 8 · Guardar el resultado
*En Excel: `Guardar como`, con nombre nuevo, nunca encima del CSV original (§11).*

Guarde `cruce` como `boletas_limpias.csv` y `clientes` como `clientes_limpios.csv`, **sin la
columna de índice**. El crudo, en internet, queda intacto — esa regla no cambia de herramienta.

In [ ]:
# SECCIÓN 8 · Guardar boletas_limpias.csv y clientes_limpios.csv
#
# Pistas (qué buscar, no qué escribir):
#   · Busque `.to_csv(nombre, index=False)`. ¿Qué pasa si olvida `index=False`?
#   · En Colab el archivo queda en la sesión (ícono de carpeta a la izquierda). Para bajarlo a su
#     computador: `from google.colab import files` y `files.download(nombre)`.

# 👉 Su código aquí

In [ ]:
# ✔ Verificación · Sección 8
verificar("archivo_guardado", os.path.exists("boletas_limpias.csv") and os.path.exists("clientes_limpios.csv"))

> ✍️ **Bitácora · Sección 8**
>
> - **Decisión que tomé:** …
> - **Por qué:** …
> - **Fuente consultada** (documentación, un compañero, IA — cuál y para qué): …
> - **Qué no funcionó a la primera:** …

---
# Parte II · Tres preguntas que en Excel no se respondieron

Hasta aquí repitió lo que ya sabía hacer. Estas tres preguntas exigen **decidir y justificar**.
No tienen una única respuesta correcta en la parte de criterio, pero los números sí se verifican.

## Pregunta G1 · ¿Primera o última copia?

En la Sección 3 eliminó los duplicados de clientes conservando la **primera** aparición
(`keep="first"`). Excel hace lo mismo y no pregunta. Pero ¿da lo mismo?

Vuelva a cargar `clientes.csv` crudo en `clientes_crudo` (la celda ya lo hace), normalice el RUT
otra vez, y construya **dos** versiones deduplicadas: `keep="first"` y `keep="last"`. Mire qué
`id_cliente` sobreviven en cada una (¿empiezan con `C0` o con `C9`?). Luego cruce las `boletas`
de venta con cada versión y cuente las huérfanas en ambos casos.

In [ ]:
# Recarga del crudo (ya escrita): la versión sin limpiar, para comparar
clientes_crudo = pd.read_csv(RUTA_DATOS + "clientes.csv")
print(clientes_crudo.shape)

In [ ]:
# PREGUNTA G1 · keep="first" vs keep="last": ¿cuántas huérfanas quedan con cada uno?
#
# Pistas (qué buscar, no qué escribir):
#   · Guarde los dos conteos en `huerfanas_first` y `huerfanas_last`.
#   · Para contar huérfanas sin volver a hacer merge: `~boletas.id_cliente.isin(ids_conservados)`
#     combinado con `boletas.id_cliente.notna()`.
#   · Compare los `id_cliente` conservados: `clientes_first.id_cliente.str[:2].value_counts()`.

# 👉 Su código aquí
huerfanas_first = ...
huerfanas_last = ...

In [ ]:
# ✔ Verificación · G1
verificar("huerfanas_keep_first", huerfanas_first)
verificar("huerfanas_keep_last", huerfanas_last)

> ✍️ **Bitácora · G1 · ¿Con cuál se queda y por qué?**
>
> - **Decisión que tomé:** …
> - **Por qué:** …
> - **Fuente consultada** (documentación, un compañero, IA — cuál y para qué): …
> - **Qué no funcionó a la primera:** …

## Pregunta G2 · ¿De dónde salen las 1.640 boletas de más?

Con `keep="last"` las huérfanas pasan de 90 a
1.730. Demuéstrelo con código: identifique los `id_cliente` de las
**45 fichas que `keep="last"` elimina** y cuente cuántas boletas
de venta apuntan a ellas. Haga lo mismo con las 45 que elimina
`keep="first"`. Después explique, **con sus palabras y en la bitácora**, qué regla general saca
de esto para cualquier limpieza futura.

In [ ]:
# PREGUNTA G2 · Cuántas boletas apuntan a las fichas que cada keep elimina
#
# Pistas (qué buscar, no qué escribir):
#   · `clientes_crudo.rut_norm.duplicated(keep="last")` marca justamente las filas que
#     `drop_duplicates(keep="last")` elimina (y lo mismo con "first").
#   · `.isin(...)` dice qué boletas apuntan a una lista de ids.
#   · Guarde en `boletas_de_eliminadas` el conteo del caso keep="last".

# 👉 Su código aquí
boletas_de_eliminadas = ...

In [ ]:
# ✔ Verificación · G2
verificar("boletas_de_fichas_eliminadas", boletas_de_eliminadas)

> ✍️ **Bitácora · G2 · La regla general**
>
> - **Decisión que tomé:** …
> - **Por qué:** …
> - **Fuente consultada** (documentación, un compañero, IA — cuál y para qué): …
> - **Qué no funcionó a la primera:** …

## Pregunta G3 · Las fechas de nacimiento imposibles

En la Sección 6 la `edad` salió con valores negativos y de más de 100 años. La causa está en el
crudo: fechas como `0001-01-01`, `1899-01-01` o
`2027-05-14`. Considere **imposible** una fecha anterior a 1920 o posterior al
30/06/2025 y cuente cuántas hay entre los 1.200 clientes.

Luego **decida usted** qué hacer con ellas — dejar la edad en blanco (`NaN`), eliminar la fila,
marcarla en una columna aparte, o algo distinto— **aplíquelo**, y reporte cuántas edades válidas
quedan. Aquí no hay número esperado: hay una decisión que debe poder defender.

In [ ]:
# PREGUNTA G3 · Fechas imposibles: contar, decidir, aplicar
#
# Pistas (qué buscar, no qué escribir):
#   · `nac = pd.to_datetime(clientes.fecha_nacimiento, errors="coerce")`; luego `nac.dt.year < 1920`
#     y `nac > pd.Timestamp("2025-06-30")`, unidos con `|`.
#   · Guarde el conteo en `fechas_imposibles`.
#   · Para dejar en blanco solo algunas celdas: `clientes.loc[condicion, "edad"] = None` (o `pd.NA`).
#   · Revise la tabla de la Sección 5 de la guía de Excel: los cuatro tratamientos siguen valiendo.

# 👉 Su código aquí
fechas_imposibles = ...
edades_validas = ...

In [ ]:
# ✔ Verificación · G3 (solo el conteo; la decisión es suya)
verificar("fechas_imposibles", fechas_imposibles)
print("edades válidas que reporta:", edades_validas)

> ✍️ **Bitácora · G3 · Qué decidió y por qué**
>
> - **Decisión que tomé:** …
> - **Por qué:** …
> - **Fuente consultada** (documentación, un compañero, IA — cuál y para qué): …
> - **Qué no funcionó a la primera:** …

---
# Parte III · Cierre

## Comparativa Excel / Colab

Complete la tabla **con lo que le pasó a usted** en este trabajo, no con generalidades.

| | Excel (01/09) | Python / Colab (hoy) |
|---|---|---|
| Cómo se expresa el trabajo | | |
| Repetir con los datos de julio | | |
| Dónde queda el registro de lo que se hizo | | |
| Cuándo se equivoca en silencio | | |
| Cuándo conviene usar cada una | | |


## Bitácora de cierre

- La decisión que **más le costó justificar** y por qué:
- Dónde usó **IA** y para qué (una línea por uso; si no la usó, dígalo):
- Qué haría distinto si mañana llegan las boletas de julio:

In [ ]:
# ✔ Resumen final · ejecute esta celda al terminar (y antes de descargar el .ipynb)
resumen()

> **Para entregar:** `Entorno de ejecución → Reiniciar y ejecutar todo`. Si termina sin errores y
> el resumen le gusta, `Archivo → Descargar → Descargar .ipynb` y súbalo al Aula Virtual antes del
> laboratorio del **martes 29/09**.

*Supermercados Diguillín es una empresa ficticia; los datos son sintéticos.*